# bronze_build_schedule_games_regular_season_only

Builds a raw-ish Bronze schedule table with one row per regular season game.

This notebook reads schedule JSON from the Bronze landing path, flattens the nested
payload enough to get to game grain, and appends the result into
`dbw_hockey_lakehouse.bronze.schedule_games`.

A few notes:
- this is ingestion-layer code, so the goal is repeatable raw handling, not heavy modeling
- write mode is append
- not fully idempotent on its own, so it should be run with controlled input dates or paired with upstream state/orchestration

## Setup

In [ ]:
from pyspark.sql import functions as F

# Source season and schedule date for this run.
# In practice this would usually come from orchestration / widgets.
season = "20242025"
schedule_date = "2024-10-01"

# Storage/account-specific values were removed from the GitHub version.
# Use your workspace config / secret scope / mount setup here instead.
storage_account = "<your_storage_account>"

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS dbw_hockey_lakehouse.bronze")

spark.sql("""
CREATE TABLE IF NOT EXISTS dbw_hockey_lakehouse.bronze.ingestion_state (
  pipeline_name STRING,
  last_success_date DATE,
  updated_at TIMESTAMP
)
USING delta
""")

# Simple seed row so there is at least one state record available.
# This notebook does not fully manage state yet, but keeping the table here
# makes the ingestion pattern easier to extend later.
spark.sql("""
INSERT INTO dbw_hockey_lakehouse.bronze.ingestion_state
SELECT 'nhl_bronze', DATE('2024-10-01'), current_timestamp()
WHERE NOT EXISTS (
  SELECT 1
  FROM dbw_hockey_lakehouse.bronze.ingestion_state
  WHERE pipeline_name = 'nhl_bronze'
)
""")

## Read raw schedule JSON

In [ ]:
schedule_path = (
    f"abfss://bronze@{storage_account}.dfs.core.windows.net/"
    f"nhl/schedule/season={season}/date={schedule_date}/schedule.json"
)

# Bronze keeps the source close to the original payload.
# We only flatten enough here to land a usable raw game-level table.
raw_df = (
    spark.read
    .option("multiLine", "true")
    .json(schedule_path)
)

raw_df.printSchema()

## Flatten to game grain

In [ ]:
games_df = (
    raw_df
    .select(F.explode("gameWeek").alias("game_week"))
    .select(
        F.col("game_week.date").alias("game_date"),
        F.explode("game_week.games").alias("game")
    )
)

## Select the Bronze game fields

In [ ]:
schedule_games_df = (
    games_df
    .select(
        F.col("game.id").alias("game_id"),
        F.to_date("game_date").alias("game_date"),
        F.col("game.startTimeUTC").alias("start_time_utc"),
        F.col("game.homeTeam.abbrev").alias("home_team"),
        F.col("game.awayTeam.abbrev").alias("away_team"),
        F.col("game.gameType").cast("int").alias("game_type"),
    )
    # Keep regular season only in this notebook.
    # gameType = 2 -> regular season
    # gameType = 1 -> preseason
    # gameType = 3 -> playoffs
    .filter(F.col("game_type") == 2)
    .withColumn("season", F.lit(season))
    .withColumn("source_date", F.lit(schedule_date))
    .withColumn("ingested_at", F.current_timestamp())
)

# This is still Bronze, so the schema stays pretty lean.
# Enough structure to be useful downstream, but not a lot of cleanup yet.

## Quick sanity checks

In [ ]:
# Basic sanity check before writing.
# Helps catch bad upstream data or a wrong source path/date.
schedule_games_df.groupBy("game_type").count().show()

schedule_games_df.selectExpr(
    "count(*) as row_count",
    "min(game_date) as min_game_date",
    "max(game_date) as max_game_date"
).show()

## Append to Bronze

In [ ]:
# Append is intentional here because Bronze is keeping raw ingestion slices over time.
# If this notebook is re-run for the same source_date without dedupe/state handling,
# duplicates are possible, so orchestration matters.
(
    schedule_games_df.write
    .format("delta")
    .mode("append")
    .saveAsTable("dbw_hockey_lakehouse.bronze.schedule_games")
)

## Verify output

In [ ]:
display(
    spark.table("dbw_hockey_lakehouse.bronze.schedule_games")
    .orderBy(F.col("ingested_at").desc(), F.col("game_date").desc())
    .limit(20)
)